In [1]:
# ─── CELL 1: Install ──────────────────────────────────────────────────────────
!pip install -q trl==0.8.6 transformers==4.41.0 peft==0.10.0 accelerate==0.29.0 datasets evaluate gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 7.9 MB/s eta 0:00:00


In [2]:
# ─── CELL 2: Imports ──────────────────────────────────────────────────────────
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

In [ ]:
# ─── CELL 3: Dataset ──────────────────────────────────────────────────────────
dataset = load_dataset("b-mc2/sql-create-context", split="train")

def format_example(row):
    return {
        "text": f"""### Task: Convert the question to SQL.

### Schema:
{row['context']}

### Question:
{row['question']}

### SQL:
{row['answer']}"""
    }

dataset = dataset.map(format_example)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset["train"] = dataset["train"].select(range(20000))
dataset["test"] = dataset["test"].select(range(1000))
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
# ─── CELL 4: Model ────────────────────────────────────────────────────────────
# codegen-350M is code-trained → much less hallucination than phi-2
# 350M params → no quantization needed, fits easily on T4
MODEL = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,ge
    torch_dtype=torch.float32,
    device_map="auto",
)
model.config.use_cache = False
model.config.use_cache = False
print("✅ Model loaded")

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

✅ Model loaded


In [ ]:
# ─── CELL 5: LoRA ─────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# ─── CELL 6: Trainer ──────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=256,
    args=TrainingArguments(
      output_dir="./sql-codegen",
      num_train_epochs=3,
      per_device_train_batch_size=8,
      gradient_accumulation_steps=4,
      warmup_steps=100,
      learning_rate=2e-4,
      fp16=False,
      bf16=False,
      logging_steps=100,
      evaluation_strategy="steps",
      eval_steps=500,
      save_strategy="steps",
      save_steps=500,
      load_best_model_at_end=True,
      report_to="none",
  ),
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# ─── CELL 7: Train (~25 min on T4) ───────────────────────────────────────────
trainer.train()

Step,Training Loss,Validation Loss
500,0.820600,0.834259
1000,0.761000,0.794298
1500,0.708500,0.779420


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=1875, training_loss=0.8007456136067709, metrics={'train_runtime': 2139.8045, 'train_samples_per_second': 28.04, 'train_steps_per_second': 0.876, 'total_flos': 1.3632915162267648e+16, 'train_loss': 0.8007456136067709, 'epoch': 3.0})

In [ ]:
# ─── CELL 8: Save ─────────────────────────────────────────────────────────────
trainer.model.save_pretrained("./sql-codegen-final")
tokenizer.save_pretrained("./sql-codegen-final")
print("✅ Model saved!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model saved!


In [ ]:
# ─── CELL 9: Push model + Deploy Space ───────────────────────────────────────
from huggingface_hub import HfApi, login

login(token="YOUR_HF_TOKEN_HERE")
api = HfApi()

# ✅ Step 1: Push fine-tuned LoRA weights FIRST
trainer.model.push_to_hub("moumita-29/deepseek-sql-qlora")
tokenizer.push_to_hub("moumita-29/deepseek-sql-qlora")
print("✅ Model pushed!")

# ✅ Step 2: Deploy Space with correct app.py
api.create_repo(
    repo_id="moumita-29/deepseek-sql-demo",  # ✅ renamed from codegen
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)
api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py",
                repo_id="moumita-29/deepseek-sql-demo", repo_type="space")
api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt",
                repo_id="moumita-29/deepseek-sql-demo", repo_type="space")
print("✅ Space deployed! Visit: https://huggingface.co/spaces/moumita-29/deepseek-sql-demo")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Done! Visit: https://huggingface.co/spaces/moumita-29/codegen-sql-qlora-demo
Build takes 3-5 minutes. Watch the Logs tab.


In [ ]:
# ─── BLEU Score Evaluation ────────────────────────────────────────────────────
!pip install -q evaluate

from evaluate import load
import torch
import re

bleu = load("bleu")

def clean_sql(raw_sql):
    noise_keywords = ["INTERSECT", "UNION", "EXCEPT", "HAVING", "OFFSET", "GROUP BY"]
    raw_upper = raw_sql.upper()
    cut_pos = len(raw_sql)
    for keyword in noise_keywords:
        pos = raw_upper.find(keyword)
        if pos != -1 and pos < cut_pos:
            cut_pos = pos
    sql = raw_sql[:cut_pos].strip()
    sql = re.sub(r"[\s,;)]+$", "", sql)
    return sql + ";"

def generate_sql(question, schema):
    prompt = f"""### Task: Convert the question to SQL.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    raw_sql = full.split("### SQL:")[-1]
    raw_sql = raw_sql.split("###")[0]
    raw_sql = " ".join(raw_sql.split()).strip()
    return clean_sql(raw_sql)

# Evaluate on 200 test examples
predictions, references = [], []
for row in dataset["test"].select(range(200)):
    pred = generate_sql(row["question"], row["context"])
    predictions.append(pred)
    references.append([row["answer"]])

score = bleu.compute(predictions=predictions, references=references)
print(f"✅ BLEU Score: {score['bleu']:.4f}")

✅ BLEU Score: 0.4516
